# Run Match

In [1]:
%load_ext autoreload
from IPython.core.display import display, HTML
display(HTML("<style>.container { width:100`% !important; }</style>"))
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)
warnings.simplefilter(action='ignore', category=UserWarning)

In [2]:
from timeutils import Timestat
from master import MasterParams, MasterDBs, MasterPaths, MasterMetas, MusicDBPermDir
from musicdb import PanDBIO
from gate import MusicDBGate
from pandas import DataFrame, Series, concat
import musicdb
from ioutils import HTMLIO, FileIO
from listUtils import getFlatList
import swifter
import dask.dataframe as dd
from match import MatchDBDataIO, AlbumReq, NameReq, MatchReq, MatchDB, CrossMatchDB, PanDBMatch
io = FileIO()

In [3]:
baseReqs = {"MetalArchives": 4,
            "RateYourMusic": 3,
            "Beatport": 35,
            "Spotify": 20,
            "Discogs": 3,  ## 12
            "Traxsource": 100000}
#baseDB    = "RateYourMusic"  # 3
#baseDB    = "MetalArchives"
#baseDB    = "Beatport"
#baseDB    = "Discogs"
#baseDB    = "Spotify"
baseDB    = "Traxsource"

minL = 15
maxL = 5000000

minA = 20
maxA = 1000000

#baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1))}
baseReq   = {baseDB: MatchReq(NameReq(min=minL, max=maxL), AlbumReq(min=minA, max=maxA))}
#baseReq   = {baseDB: AlbumReq(min=baseReqs.get(baseDB), max=baseReqs.get(baseDB)+1)}
#baseReq   = {baseDB: AlbumReq(min=10, max=baseReqs.get(baseDB,10000)+1, rnd=10000)}

compareDBs  = ["RateYourMusic", "LastFM", "Spotify", "Genius", "Discogs", "MusicBrainz", "Deezer", "MetalArchives"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Genius", "Discogs", "MusicBrainz", "Beatport"] # "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Spotify", "Discogs", "MusicBrainz", "Beatport", "Traxsource"] # "LastFM", "Deezer"]
compareDBs  = ["RateYourMusic", "Traxsource", "Spotify", "Beatport"]
compareReqs = {compareDB: MatchReq(NameReq(min=minL-5, max=maxL+5), AlbumReq(min=3)) for compareDB in compareDBs if compareDB not in [baseDB]}
compareDBs  = list(compareReqs.keys())

matchReqs  = {**baseReq, **compareReqs}
mediaTypes = ["Album", "SingleEP"]
mediaTypes = ["{0}Media".format(mediaType) for mediaType in mediaTypes]
mediaTypes = list(MasterMetas().getMedias().values())
reqs       = {"Media": mediaTypes, "Reqs": matchReqs, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Primary Run Params:")
print("  ==> DBs:   [{0}] x {1}]".format(baseDB,list(compareReqs.keys())))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(reqs["Match"]))

crossreqs  = {"Media": mediaTypes, "Reqs": {baseDB: MatchReq(AlbumReq(min=2))}, "Mask": baseDB, "NPart": 3, "Match": {"Artist": 0.85, "Medium": 2, "Tight": 1}}
print("Cross Match Run Params:")
print("  ==> DBs:   {0} x [{1}]".format(list(compareReqs.keys()), baseDB))
print("  ==> Media: {0}".format(mediaTypes))
print("  ==> Match: {0}".format(crossreqs["Match"]))

Primary Run Params:
  ==> DBs:   [Traxsource] x ['RateYourMusic', 'Spotify', 'Beatport']]
  ==> Media: ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Match: {'Artist': 0.85, 'Medium': 2, 'Tight': 1}
Cross Match Run Params:
  ==> DBs:   ['RateYourMusic', 'Spotify', 'Beatport'] x [Traxsource]
  ==> Media: ['Album', 'SingleEP', 'Appearance', 'Technical', 'Mix', 'Bootleg', 'AltMedia', 'Other']
  ==> Match: {'Artist': 0.85, 'Medium': 2, 'Tight': 1}


In [4]:
baseIO = MatchDBDataIO(db=baseDB, mediaTypes=reqs["Media"], mask=reqs["Mask"], verbose=False, base=True)
baseIO.loadNames()
baseIO.setAvailableNames(reqs["Reqs"][baseDB])
del baseIO

     ==> Cut[Media]            70161 Artists With [2/500] Min/Max
     ==> Cut[MediaLen]         16646 Artists With [20/500] Min/Max
     ==> Cut[Names]            70161 Artists With [1/109] Min/Max
     ==> Cut[NamesLen]         12435 Artists With [15/109] Min/Max
     ==> Cut[Final]             1683 Artists With [/] Min/Max


# Match & Cross Match

In [5]:
mdb = MatchDB(baseDB=baseDB, compareDBs=compareDBs, reqs=reqs)
mdb.match()
mdb.save()
del mdb

*******************************************************************************************************************************************************************************
*                                                                       MatchDB()                                                                                             *
*******************************************************************************************************************************************************************************
Process [Matching [Traxsource] Against ['RateYourMusic', 'Spotify', 'Beatport']] Start    ==> Time Is 2022-05-03 08:26:52

------------------------------------------------------------------------------------------------------------------------------------------------------
---------------------------------------------------------------------- Traxsource --------------------------------------------------------------------
  Process [Loading Artist Names] Start    ==> T

100%|██████████| 561/561 [00:17<00:00, 32.26it/s]


  Process [String Matching 1683 [Traxsource] x 29615 [RateYourMusic] Artist Names] Ran For 25 Seconds    ==> Time Is 2022-05-03 08:27:21
     ==> Found 1683 Name Results
     ==> Found 163 Artists With One Or More Matches
     ==> Found 173 Possible Matches
    Process [Loading Traxsource Media Data] Start    ==> Time Is 2022-05-03 08:27:21
    Process [Loading Traxsource Media Data] Ran For 8 Seconds    ==> Time Is 2022-05-03 08:27:29
    Process [Loading RateYourMusic Media Data] Start    ==> Time Is 2022-05-03 08:27:29
    Process [Loading RateYourMusic Media Data] Ran For 2 Seconds    ==> Time Is 2022-05-03 08:27:31
  Process [String Matching 173 [Traxsource] Album Names] Start    ==> Time Is 2022-05-03 08:27:31
      Process [Matching 173 Artists' Albums] Start    ==> Time Is 2022-05-03 08:27:31
      Process [Matching 173 Artists' Albums] Ran For 4 Seconds    ==> Time Is 2022-05-03 08:27:35
  Process [String Matching 173 [Traxsource] Album Names] Ran For 5 Seconds    ==> Time Is 

100%|██████████| 561/561 [00:49<00:00, 11.41it/s]


  Process [String Matching 1683 [Traxsource] x 84611 [Spotify] Artist Names] Ran For 1 Minute and 10 Seconds    ==> Time Is 2022-05-03 08:28:52
     ==> Found 1683 Name Results
     ==> Found 447 Artists With One Or More Matches
     ==> Found 514 Possible Matches
    Process [Loading Traxsource Media Data] Start    ==> Time Is 2022-05-03 08:28:52
    Process [Loading Traxsource Media Data] Ran For     ==> Time Is 2022-05-03 08:28:52
    Process [Loading Spotify Media Data] Start    ==> Time Is 2022-05-03 08:28:52
    Process [Loading Spotify Media Data] Ran For 7 Seconds    ==> Time Is 2022-05-03 08:28:59
  Process [String Matching 514 [Traxsource] Album Names] Start    ==> Time Is 2022-05-03 08:28:59
      Process [Matching 514 Artists' Albums] Start    ==> Time Is 2022-05-03 08:28:59
      Process [Matching 514 Artists' Albums] Ran For 13 Seconds    ==> Time Is 2022-05-03 08:29:12
  Process [String Matching 514 [Traxsource] Album Names] Ran For 17 Seconds    ==> Time Is 2022-05-03 0

100%|██████████| 561/561 [00:41<00:00, 13.49it/s]


  Process [String Matching 1683 [Traxsource] x 75307 [Beatport] Artist Names] Ran For 1 Minute and 6 Seconds    ==> Time Is 2022-05-03 08:30:24
     ==> Found 1683 Name Results
     ==> Found 936 Artists With One Or More Matches
     ==> Found 1188 Possible Matches
    Process [Loading Traxsource Media Data] Start    ==> Time Is 2022-05-03 08:30:25
    Process [Loading Traxsource Media Data] Ran For     ==> Time Is 2022-05-03 08:30:25
    Process [Loading Beatport Media Data] Start    ==> Time Is 2022-05-03 08:30:25
    Process [Loading Beatport Media Data] Ran For 4 Seconds    ==> Time Is 2022-05-03 08:30:29
  Process [String Matching 1188 [Traxsource] Album Names] Start    ==> Time Is 2022-05-03 08:30:29
      Process [Matching 1188 Artists' Albums] Start    ==> Time Is 2022-05-03 08:30:29
      Process [Matching 1188 Artists' Albums] Ran For 17 Seconds    ==> Time Is 2022-05-03 08:30:46
  Process [String Matching 1188 [Traxsource] Album Names] Ran For 25 Seconds    ==> Time Is 2022-

In [6]:
mdbpd = MusicDBPermDir()
mres  = io.get(mdbpd.getMatchPermPath().join("primaryMatch.p"))
cmdb  = CrossMatchDB(baseDB, mres, crossreqs, verbose=True)
cmdb.match()
cmdb.save()

del cmdb

*******************************************************************************************************************************************************************************
*                                                                       MatchDB()                                                                                             *
*******************************************************************************************************************************************************************************
Process [Cross Matching [Traxsource] Against ['RateYourMusic', 'Spotify', 'Beatport']] Start    ==> Time Is 2022-05-03 08:30:54
     ==> Cut[Media]            70161 Artists With [2/500] Min/Max
     ==> Cut[MediaLen]         70161 Artists With [2/500] Min/Max
     ==> Cut[Final]            70161 Artists With [/] Min/Max
  Process [String Matching 1362 ['RateYourMusic', 'Spotify', 'Beatport'] x 70161 [Traxsource] Artist Names] Start    ==> Time Is 2022-05-03 08:30:56


100%|██████████| 454/454 [00:28<00:00, 16.00it/s]


     ==> Found 1362 Name Results
     ==> Found 1362 Artists With One Or More Matches
     ==> Found 1895 Possible Matches
  Process [String Matching 1362 ['RateYourMusic', 'Spotify', 'Beatport'] x 70161 [Traxsource] Artist Names] Ran For 47 Seconds    ==> Time Is 2022-05-03 08:31:44
  Process [String Matching 1895 ['RateYourMusic', 'Spotify', 'Beatport'] Album Names] Start    ==> Time Is 2022-05-03 08:31:44
      Process [Matching 1895 Artists' Albums] Start    ==> Time Is 2022-05-03 08:31:44
      Process [Matching 1895 Artists' Albums] Ran For 27 Seconds    ==> Time Is 2022-05-03 08:32:11
  Process [String Matching 1895 ['RateYourMusic', 'Spotify', 'Beatport'] Album Names] Ran For 39 Seconds    ==> Time Is 2022-05-03 08:32:23
Saving Primary Match Results To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdbmatch/crossMatch.p
==== Match Results ====
By DB:
Traxsource    1656
Name: DB, dtype: int64
By Quality:
Pure     1515
Great      57
Sole       40
Poor       15
Good       13
Near 

In [7]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=5)
pdbm.master()
pdbm.merge(allownew=True, verbose=False)

MatchID()
5f14fb8cff   | 100254                   Beatport       242590                                  8    JULIAN RODRIGUEZ                                  JULIAN RODRIGUEZ                                   | 5f14fb8cff
7071c7b1fd   |                          Spotify        7kEsMIqwo5MIbKpeD4x1MF                  8    JULIAN RODRIGUEZ                                  JULIAN RODRIGUEZ                                   | 7071c7b1fd
ce02aefbd6   | 100271                   Beatport       12449                                   8    ARIEL PERAZZOLI                                   ARIEL PERAZZOLI                                    | ce02aefbd6
e806f758fa   |                          Spotify        4uxHVwlQA0CQXTh9DMMkip                  8    ARIEL PERAZZOLI                                   ARIEL PERAZZOLI                                    | e806f758fa
df7d9778c6   | 10054                    Spotify        5b5Cy5zyMcaPwBnbSNNic1                  8    CHRISTIAN QUAST                   

74dfc24c20   | 158339                   Spotify        6tZxUxheS7w3953cQFOXkd                  6    ADALBERTO SANTIAGO                                ADALBERTO SANTIAGO                                 | 74dfc24c20
c7bd9cafce   | 159593                   Beatport       81255                                   8    CAMO &AMP; KROOKED                                CAMO & KROOKED                                     | c7bd9cafce
cde3dbe78f   |                          RateYourMusic  485431                                  8    CAMO &AMP; KROOKED                                CAMO & KROOKED                                     | cde3dbe78f
8492441921   |                          Spotify        2N8IPNZTiNo3nj4mreOlHU                  8    CAMO &AMP; KROOKED                                CAMO & KROOKED                                     | 8492441921
150eb4b86a   | 159598                   Beatport       183719                                  8    GESLOTEN CIRKEL                             

507cfbb532   | 26459                    Beatport       59371                                   8    GROOVE PHENOMENON                                 GROOVE PHENOMENON                                  | 507cfbb532
7a8aa189f3   | 26696                    Beatport       108484                                  8    ILIAS KATELANOS                                   ILIAS KATELANOS                                    | 7a8aa189f3
7558333d64   |                          Spotify        1sqLCBbu9hQESkdUn8hpfC                  8    ILIAS KATELANOS                                   ILIAS KATELANOS                                    | 7558333d64
5c2509d8ff   | 268535                   Beatport       5301                                    8    SEBASTIAN MOORE                                   SEBASTIAN MOORE                                    | 5c2509d8ff
5b92149f75   | 269051                   Beatport       511281                                  8    BOSSA DEL CHILL                             

07d54ef004   | 419781                   Beatport       655425                                  8    AGENT ORANGE DJ                                   AGENT ORANGE DJ                                    | 07d54ef004
9c9c5fca12   | 42134                    Beatport       168752                                  8    ADRIAN IZQUIERDO                                  ADRIAN IZQUIERDO                                   | 9c9c5fca12
3d2364ce4f   | 4226                     Spotify        3WcLzycV4ilVobSdUBKn9M                  8    JEPHTE GUILLAUME                                  JEPHTÉ GUILLAUME                                  | 3d2364ce4f
ce4ae4f9d9   | 4227                     Beatport       10242                                   8    TERRENCE PARKER                                   TERRENCE PARKER                                    | ce4ae4f9d9
7e123ee5d8   |                          RateYourMusic  33809                                   8    TERRENCE PARKER                             

ae5701c9fa   | 65795                    Beatport       197946                                  8    ANDREY DJACKONDA                                  ANDREY DJACKONDA                                   | ae5701c9fa
fb2d5d7b01   | 66850                    Beatport       124603                                  8    MIKAEL PFEIFFER                                   MIKAEL PFEIFFER                                    | fb2d5d7b01
14c867be6c   | 6755                     Beatport       5590                                    8    WIGHNOMY BROTHERS                                 WIGHNOMY BROTHERS                                  | 14c867be6c
a38b1b07a5   |                          RateYourMusic  152164                                  8    WIGHNOMY BROTHERS                                 WIGHNOMY BROTHERS                                  | a38b1b07a5
d6f09249a0   | 68331                    RateYourMusic  692515                                  8    FATIMA AL QADIRI                            

  ==> Found [730] Known Matches
  ==> Found [3] Multi Matches
  ==> Found [28] New Matches
  ==> Added [730] Known Matches
  ==> Added [28] New Matches
Saving Master DataFrame To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/musicdb/manualEntries.p
  ==> Saving [3] Multi Match Artists' Data To /Users/tgadfort/opt/anaconda3/envs/py310/pandb/mdbmatch/multiMatch.p


# Extra Match

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=4, maxQual=5)

In [ ]:
pdbm.include("""
939277b92c
621f283f66
7699b8dcdd
2316f9bbe1
b681a1a1ad
1facf7567f
ddf698cc67
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=2, maxQual=4)

In [ ]:
pdbm.include("""
55029a0d84
""")
pdbm.master()
pdbm.merge()

In [ ]:
pdbm = PanDBMatch(baseDB, verbose=True)
pdbm.select(minQual=1, maxQual=2)

In [ ]:
pdbm.include("""
7d66c0bc49
b9b71d5d48
fd44e5bc68
60937e061e
967ce81174
51fc14c777
""")

pdbm.master()
pdbm.merge()

In [ ]:
del pdbm

In [ ]:
# Fix This:
#afc404588f   | 183166                   Discogs        113700                                  2    STEVE MASTERSON                                   STEVE MAESTRO                                      | afc404588f

In [ ]:
pdbio = PanDBIO()
mmeDF = pdbio.getData()